# Sfoglia gli eventi di un run

Una pagina per evento: un grafico per canale, 2 righe per 8 colonne. Si cambia solo la cella
**CONFIGURAZIONE**, poi *Run All*: nell'ultima cella compare una casella di testo, e **Invio** mostra
l'evento successivo. Un numero porta a quell'evento, `b` torna indietro, `q` esce.

Legge un evento per volta (anche dai run da qualche GB, dai `.h5.gz` e da un run ancora in corso), e
disegna i segnali con la loro polarita' vera. Come funziona: in fondo.

In [ ]:
# ===================== CONFIGURAZIONE =====================
FILE    = None      # percorso del run (.h5 o .h5.gz); None = il piu' recente nella OutputDir di config/run-local.toml
EVENTO  = 0         # da quale evento si parte

RIGHE, COLONNE = 2, 8       # 16 grafici, un canale ciascuno
UNITA          = "counts"   # "counts" (conteggi ADC) oppure "mV"
TOGLI_PIEDISTALLO = False   # True: ogni traccia meno la mediana dei primi PIEDISTALLO_NS
PIEDISTALLO_NS    = 200.0
STESSA_SCALA_Y    = False   # True: lo stesso intervallo verticale su tutti i canali
DIMENSIONE        = (19, 6.5)   # pollici
SPESSORE, COLORE  = 0.7, "#1f4e9c"

In [ ]:
## === Preparazione: il file e le funzioni ===
import glob
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# La radice del repository si cerca risalendo dalla cartella del notebook (VS Code
# la dice in __vsc_ipynb_file__) o, altrimenti, da quella del kernel.
_partenza = os.path.dirname(globals().get("__vsc_ipynb_file__", "")) or os.getcwd()
ROOT = os.path.abspath(_partenza)
while not os.path.exists(os.path.join(ROOT, "tools", "daqio.py")):
    if os.path.dirname(ROOT) == ROOT:
        raise RuntimeError("Non trovo il repository daq-standalone risalendo da %s." % _partenza)
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, "tools"))
import daqio                                                      # noqa: E402
from sfoglia_eventi import apri, passo_ns, mv_per_conteggio      # noqa: E402  stessa lettura dello script


def ultimo_run():
    """Il file di dati piu' recente nella OutputDir di config/run-local.toml."""
    import tomllib
    toml = os.path.join(ROOT, "config", "run-local.toml")
    if not os.path.exists(toml):
        raise RuntimeError("FILE = None, ma qui non c'e' config/run-local.toml: scrivi il percorso in FILE.")
    with open(toml, "rb") as f:
        cartella = tomllib.load(f).get("digitizer", {}).get("OutputDir", os.path.join(ROOT, "data"))
    file = glob.glob(os.path.join(cartella, "*.h5")) + glob.glob(os.path.join(cartella, "*.h5.gz"))
    if not file:
        raise RuntimeError("Nessun run in %s." % cartella)
    return max(file, key=os.path.getmtime)


# Un file aperto dal giro prima si chiude: rieseguire la cella non lascia file aperti.
if "_f" in globals():
    try:
        _f.close()
        daqio._cleanup(_tmp)
    except Exception:
        pass

percorso = FILE or ultimo_run()
_f, _tmp, vivo = apri(percorso)
hdr = daqio.read_header(_f)
ds = _f["/events/waveforms"]
canali = [int(c) for c in np.atleast_1d(hdr["ChannelList"])]
ns, ncan = int(hdr["SamplesPerChannel"]), len(canali)
dt = passo_ns(hdr)
t = np.arange(ns) * dt
conv = mv_per_conteggio(hdr, ncan) if UNITA == "mV" else np.ones(ncan)
n_ped = max(1, min(ns, int(PIEDISTALLO_NS / dt)))
contatore = _f["/events/event_counter"] if "/events/event_counter" in _f else None
tempo = _f["/events/timestamp"] if "/events/timestamp" in _f else None
unita_t = float(tempo.attrs.get("UnitNs", 8.0)) if tempo is not None else None


def n_eventi():
    if vivo:
        ds.refresh()
    return ds.shape[0]


def disegna(i):
    """Un evento: un grafico per canale, segnali con la loro polarita' vera."""
    w = ds[i].reshape(ncan, ns).astype(float)
    if TOGLI_PIEDISTALLO:
        w -= np.median(w[:, :n_ped], axis=1, keepdims=True)
    w *= conv[:, None]
    fig, assi = plt.subplots(RIGHE, COLONNE, figsize=DIMENSIONE, sharex=True)
    assi = np.atleast_1d(assi).ravel()
    ylab = ("mV" if UNITA == "mV" else "ADC counts") + (" - pedestal" if TOGLI_PIEDISTALLO else "")
    for k, ax in enumerate(assi):
        if k >= ncan:
            ax.set_visible(False)
            continue
        ax.plot(t, w[k], lw=SPESSORE, color=COLORE)
        ax.set_title("ch%d" % canali[k], fontsize=9, loc="left")
        ax.grid(alpha=0.3)
        ax.tick_params(labelsize=7)
        if k % COLONNE == 0:
            ax.set_ylabel(ylab, fontsize=8)
        if k >= (RIGHE - 1) * COLONNE or k + COLONNE >= ncan:
            # L'ultimo grafico visibile della colonna porta l'asse dei tempi: con
            # sharex i numeri restano solo sulla riga in basso, che con meno di
            # 16 canali e' vuota.
            ax.set_xlabel("time [ns]", fontsize=8)
            ax.tick_params(labelbottom=True)
    if STESSA_SCALA_Y:
        lo, hi = w[:min(ncan, len(assi))].min(), w[:min(ncan, len(assi))].max()
        for ax in assi[:ncan]:
            ax.set_ylim(lo - 0.05 * (hi - lo or 1), hi + 0.05 * (hi - lo or 1))
    titolo = "%s  -  event %d of %d" % (os.path.basename(percorso), i, n_eventi())
    if contatore is not None:
        if vivo:
            contatore.refresh()
        titolo += "   trigger id %d" % int(contatore[i])
    if tempo is not None:
        if vivo:
            tempo.refresh()
        titolo += "   t = %.6f s" % (int(tempo[i]) * unita_t * 1e-9)
    fig.suptitle(titolo, fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig


print("%s: %d events, %d channels %s, %d samples of %.3g ns%s"
      % (os.path.basename(percorso), n_eventi(), ncan, canali, ns, dt, "  (run in progress)" if vivo else ""))
if ncan > RIGHE * COLONNE:
    print("Attenzione: si vedono i primi %d canali (RIGHE x COLONNE)." % (RIGHE * COLONNE))

In [ ]:
## === Sfoglia: Invio = evento successivo, numero = vai a quell'evento, b = indietro, q = esci ===
i = min(max(0, EVENTO), n_eventi() - 1)
while True:
    clear_output(wait=True)
    disegna(i)
    plt.show()
    r = input("[event %d of %d]  Enter = next, number = go to, b = back, q = quit: "
              % (i, n_eventi() - 1)).strip().lower()
    if r == "q":
        break
    if r == "b":
        i = max(0, i - 1)
    elif r.isdigit():
        i = min(int(r), n_eventi() - 1)
    elif i + 1 < n_eventi():
        i += 1
    else:
        print("Last event of the file%s." % (": the run is still going on, Enter to look again" if vivo else ""))
print("Stopped at event %d. Run this cell again to go on from EVENTO, or change EVENTO." % i)

## Come funziona

- Il file si apre con `daqio` (lo stesso lettore del monitor e degli altri notebook) e si legge **un
  evento alla volta** dal dataset `/events/waveforms`: un run da 3 GB non viene caricato in memoria.
  Un `.h5.gz` si decomprime prima in un file temporaneo; un run ancora in corso si apre in SWMR, e
  arrivati in fondo Invio guarda se sono arrivati eventi nuovi.
- I canali sono quelli di `ChannelList` nel file, nell'ordine in cui la DAQ li ha scritti: per il
  VX2730B 0-15, per il V1742 quelli registrati (meno di 16 grafici: gli altri restano vuoti).
- `UNITA = "mV"` usa `ADCToVolts` del file, canale per canale e col guadagno (VX2730B); per il V1742,
  che non lo scrive, 1 V su 12 bit.
- La lettura e' la stessa di `tools/sfoglia_eventi.py`, che fa la stessa cosa da terminale in una
  finestra di matplotlib.